In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from functools import partial

import numpy as np
from astropy import units as u
from astropy.coordinates import ICRS
from astropy.table import QTable
from astropy_healpix import HEALPix
from IPython.display import Markdown, display
from ligo.skymap import plot  # noqa: F401
from ligo.skymap.util import progress_map_vectorized
from m4opt.fov import footprint_healpix
from m4opt.missions import uvex as mission
from m4opt.synphot import observing
from m4opt.utils.numpy import count_intersect1d_combinations, intersect1d
from matplotlib import colors
from matplotlib import pyplot as plt
from regions import Regions
from survey import survey_programs
from synphot import ConstFlux1D, SourceSpectrum
from tqdm.auto import tqdm


def sky_fraction_area_axes(ax: plt.Axes, cumulative: bool = False):
    """Configure sky fraction and linked sky area axes."""
    factor = u.spat.to(1e3 * u.deg**2)

    def transform(fraction):
        return fraction * factor

    def inverse(area):
        return area / factor

    if cumulative:
        label = "Cumulative sky"
    else:
        label = "Sky"
    ax.set_ylabel(f"{label} fraction")
    return ax.secondary_yaxis(
        "right", (transform, inverse), ylabel=f"{label} area / $10^3$ deg$^2$"
    )


def fast_ecdf(
    a: np.ndarray[np.tuple[int]],
    *,
    norm: float = 1,
    points: int = 1000,
    ax: plt.Axes | None = None,
    **kwargs,
):
    """
    Plot a downsampled empirical cumulative distribution function.

    This is similar to :meth:`matplotlib.pyplot.ecdf` but will render faster
    and produce smalller PDFs when given very large datasets.
    """
    if ax is None:
        ax = plt.gca()
    quantiles = np.linspace(0, 1, points)
    abscissae = np.quantile(a, quantiles)
    return ax.plot(abscissae, quantiles * norm, **kwargs)

In [ ]:
(bounding_rectangle,) = Regions.read("../fov/bounding-rectangle.ds9")
(inscribed_circle,) = Regions.read("../fov/inscribed-circle.ds9")
plan = QTable.read("../tables/plan.ecsv")

## Time usage

In [ ]:
total_time = u.Quantity(list(plan.meta["total_time"].values())).to(u.day)
action = list(plan.meta["total_time"].keys())
plt.pie(
    total_time,
    labels=action,
    autopct=lambda pct: (0.01 * pct * total_time.sum().to(u.day)).round(2),
)
plt.savefig("../visualizations/time-utilization.pdf", metadata={"CreationDate": None})

## Fraction of sky visited N times

In [ ]:
obs = plan[plan["action"] == "observe"]
hpx = HEALPix(nside=2048, frame=ICRS())

fovs = [bounding_rectangle, inscribed_circle, mission.fov]
fov_names = ["Bounding rectangle", "Inscribed circle", "Chips"]
visits = []
visit_maps = []

for fov in fovs:
    footprints = progress_map_vectorized(
        partial(footprint_healpix, hpx, fov),
        obs["target_coord"].unmasked,
        obs["roll"].unmasked,
        jobs=None,
    )
    visit_maps.append(
        visit_map := np.bincount(np.concatenate(footprints), minlength=hpx.npix)
    )
    visits.append(np.bincount(visit_map))
max_visits = max(len(v) for v in visits)
visits = np.stack([np.pad(v, (0, max_visits - len(v))) for v in visits])

In [ ]:
survey_footprints = [
    np.arange(hpx.npix)
    if program.region is None
    else footprint_healpix(hpx, program.region)
    for program in survey_programs
]

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
nrows = len(fov_names)
fig, axs = plt.subplots(
    nrows,
    1,
    figsize=(fig_width, 0.6 * nrows * fig_height),
    dpi=300,
    subplot_kw={"projection": "astro aitoff", "center": "8h 0d"},
)
for ax, fov_name, visit_map in zip(axs, fov_names, visit_maps):
    im = ax.imshow_hpx(
        visit_map, norm=colors.LogNorm(vmin=0.8, vmax=max_visits, clip=True)
    )
    ax.set_title(fov_name)
    ax.grid()
    fig.colorbar(im, ax=ax).set_label("Number of visits")
plt.savefig("../visualizations/visit-map.pdf", metadata={"CreationDate": None})

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
nrows = len(fov_names)
bins = np.concatenate(
    [[0]]
    + [
        np.arange(1, 10) * 10**order
        for order in range(int(np.ceil(np.log10(max_visits))))
    ]
)
bins = bins[bins <= max_visits]
fig, axs = plt.subplots(
    nrows, 1, figsize=(fig_width, nrows * fig_height), sharex=True, sharey=True
)
for ax, fov_name, visit in zip(axs, fov_names, visits):
    ax.hist(np.arange(len(visit)), bins=bins, weights=visit / hpx.npix)
    ax.grid(which="both")
    ax.set_ylabel("Sky fraction")
    ax.set_title(fov_name)
    sky_fraction_area_axes(ax)
ax.set_xlabel("Number of visits")
ax.set_ylim(0, None)
ax.set_xscale("log")
plt.savefig("../visualizations/visit-distribution.pdf", metadata={"CreationDate": None})

## Limiting magnitude map

In [ ]:
source_spectrum = SourceSpectrum(ConstFlux1D, amplitude=0 * u.ABmag)
dwell = 900 * u.s
snr = 5

In [ ]:
# The HEALPix field footprints are ragged in length.
# Store them in a masked array so that we can broadcast over time and HEALPix index.
shape = (max(len(footprint) for footprint in footprints), len(footprints))
ipix = np.zeros(shape, dtype=np.intp)
mask = np.zeros(shape, dtype=bool)
for i, footprint in enumerate(footprints):
    ipix[: len(footprint), i] = footprint
    mask[len(footprint) :, i] = True
observer_location = obs["observer_location"].unmasked
obstime = obs["start_time"].unmasked
target_coord = hpx.healpix_to_skycoord(np.arange(hpx.npix))[ipix]

In [ ]:
# The pixel array is too lage to process all at once without running out of
# memory, so process it in chunks.
chunk_size = 5000
num_chunks = len(obstime) // chunk_size

limmags_by_band = {
    bandpass: np.empty(shape) for bandpass in mission.detector.bandpasses
}
offset = 0
for i, (l, t, o) in enumerate(
    zip(
        tqdm(np.array_split(observer_location, num_chunks, axis=-1)),
        np.array_split(target_coord, num_chunks, axis=-1),
        np.array_split(obstime, num_chunks, axis=-1),
    )
):
    with observing(observer_location=l, target_coord=t, obstime=o):
        for bandpass, value in limmags_by_band.items():
            result = mission.detector.get_limmag(
                snr=snr,
                exptime=dwell,
                source_spectrum=source_spectrum,
                bandpass=bandpass,
            ).value
            value[:, offset : offset + result.shape[1]] = result
    offset += result.shape[1]

In [ ]:
# Calculate stacked limiting magnitude, assuming Gaussian noise.
zeropoint = 24  # prevent catastrophic overflow
coadded_limmags_by_band = {
    key: 1.25
    * np.log10(
        np.bincount(
            ipix[~mask],
            weights=10 ** (0.8 * (value[~mask] - zeropoint)),
            minlength=hpx.npix,
        )
    )
    + zeropoint
    for key, value in limmags_by_band.items()
}

In [ ]:
ax = plt.axes()
for key, value in coadded_limmags_by_band.items():
    # The line series has a large number of data points, one per HEALpix pixel.
    ax.ecdf(value, label=key, rasterized=True)
ax.legend(loc="upper left")
ax.set_xlabel("Coadded limiting magnitude")
sky_fraction_area_axes(ax, cumulative=True)
ax.grid()
ax.set_xlim(24, 28)
ax.set_ylim(0, 1)
plt.savefig(
    "../visualizations/limiting-magnitude-distribution.pdf",
    metadata={"CreationDate": None},
)

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
nrows = len(coadded_limmags_by_band)
fig, axs = plt.subplots(
    nrows,
    1,
    figsize=(fig_width, 0.6 * nrows * fig_height),
    dpi=300,
    subplot_kw={"projection": "astro aitoff", "center": "8h 0d"},
)
for ax, (key, value) in zip(axs, coadded_limmags_by_band.items()):
    plt.colorbar(ax.imshow_hpx(value, vmin=24, vmax=28, cmap="viridis_r")).set_label(
        "Limiting magnitude"
    )
    ax.set_title(key)
    ax.grid()
plt.savefig(
    "../visualizations/limiting-magnitude-map.pdf", metadata={"CreationDate": None}
)

## Cadence distribution

In [ ]:
start_mjd = obs["start_time"].mjd
weights = count_intersect1d_combinations(footprints)
i, j = np.triu_indices(len(start_mjd), 1)
keep = weights > 0
weights = weights[keep]
i = i[keep]
j = j[keep]
delays = start_mjd[j] - start_mjd[i]
footprint_intersections = [
    intersect1d(footprints[ii], footprints[jj])
    for ii, jj in tqdm(np.column_stack((i, j)))
]

In [ ]:
fraction_visited_at_most_once = (visits[-1][0] + visits[-1][-1]) / hpx.npix

for nbins in tqdm([2, 4, 8, 20, 40]):
    fig, axs = plt.subplots(1, 2, sharey=True, width_ratios=(20, 1))
    bins = np.logspace(
        np.floor(np.log10(delays.min())), np.ceil(np.log10(delays.max())), nbins + 1
    )
    axs[0].hist(
        delays,
        weights=weights / hpx.npix,
        bins=bins,
    )
    axs[0].set_xscale("log")
    axs[0].set_xlabel("Time delay (days)")
    axs[0].set_ylabel("Sky fraction")
    axs[0].grid()

    axs[1].bar("Visited\nat most once", fraction_visited_at_most_once)

    axs[0].spines.left.set_visible(False)
    axs[0].spines.top.set_visible(False)
    axs[0].spines.right.set_visible(False)
    axs[1].spines.left.set_visible(False)
    axs[1].spines.top.set_visible(False)
    axs[1].spines.right.set_visible(False)
    plt.setp(axs[1].xaxis.get_ticklines(), visible=False)
    plt.setp(axs[1].yaxis.get_ticklines(), visible=False)
    axs[0].set_title("Cadence distribution - including repeat visits")
    fig.savefig(
        f"../visualizations/cadence-distribution-{nbins}-bins.pdf",
        metadata={"CreationDate": None},
    )

In [ ]:
fraction_visited_at_most_once = (visit_maps[-1] <= 1).sum() / hpx.npix

for nbins in tqdm([2, 4, 8, 20, 40]):
    fig, axs = plt.subplots(1, 2, sharey=True, width_ratios=(20, 1))
    bins = np.logspace(
        np.floor(np.log10(delays.min())), np.ceil(np.log10(delays.max())), nbins + 1
    )
    bin_assignments = np.digitize(delays, bins)
    pix_in_bin = np.zeros((len(bins) + 1, hpx.npix), dtype=bool)
    for bin_assignment, footprint_intersection in zip(
        tqdm(bin_assignments), footprint_intersections
    ):
        pix_in_bin[bin_assignment, footprint_intersection] = True
    bin_pixel_counts = pix_in_bin.sum(axis=1)

    axs[0].bar(
        bins[:-1], bin_pixel_counts[1:-1] / hpx.npix, bins[1:] - bins[:-1], align="edge"
    )
    axs[0].set_xscale("log")
    axs[0].set_xlabel("Time delay (days)")
    axs[0].set_ylabel("Sky fraction")
    axs[0].grid()

    axs[1].bar("Visited\nat most once", fraction_visited_at_most_once)

    axs[0].spines.left.set_visible(False)
    axs[0].spines.top.set_visible(False)
    axs[0].spines.right.set_visible(False)
    axs[1].spines.left.set_visible(False)
    axs[1].spines.top.set_visible(False)
    axs[1].spines.right.set_visible(False)
    plt.setp(axs[1].xaxis.get_ticklines(), visible=False)
    plt.setp(axs[1].yaxis.get_ticklines(), visible=False)
    axs[0].set_title("Cadence distribution - excluding repeat visits")
    fig.savefig(
        f"../visualizations/cadence-distribution-no-repeats-{nbins}-bins.pdf",
        metadata={"CreationDate": None},
    )

## Slew distribution

In [ ]:
downlink_slew_angles = plan["slew_angle"][1:-1][
    ((plan[:-2]["action"] == "downlink") | (plan[2:]["action"] == "downlink"))
    & (plan[1:-1]["action"] == "slew")
].filled(np.nan)
non_downlink_slew_angles = plan["slew_angle"][1:-1][
    (plan[:-2]["action"] != "downlink")
    & (plan[2:]["action"] != "downlink")
    & (plan[1:-1]["action"] == "slew")
].filled(np.nan)

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
fig, axs = plt.subplots(
    2,
    1,
    figsize=(fig_width, 2 * fig_height),
    sharex=True,
    gridspec_kw={"hspace": 0.1},
)
axs[0].hist(
    non_downlink_slew_angles.to_value(u.deg), bins=np.arange(0, 181, 1), log=True
)
axs[0].set_ylabel("Observing slews only)")
axs[1].set_ylabel("Downlink slews only)")
axs[1].hist(downlink_slew_angles.to_value(u.deg), bins=np.arange(0, 181, 15), log=True)
axs[1].set_xlabel("Slew angle (deg)")
axs[1].set_xlim(0, 180)
axs[1].xaxis.set_major_locator(plt.MultipleLocator(15))
fig.suptitle("Slew angle frequency distribution")
fig.savefig(
    "../visualizations/slew-angle-distribution.pdf",
    metadata={"CreationDate": None},
)

In [ ]:
survey_footprints_by_name = {
    program.name: survey_footprint
    for program, survey_footprint in zip(survey_programs, survey_footprints)
}

In [ ]:
display(Markdown("## LMLZ Wide"))

display(
    QTable(
        rows=[
            {
                "description": "Area",
                "requirement": 7500,
                "value": (visit_map[survey_footprints_by_name["lmlz_wide"]] > 0).sum()
                * hpx.pixel_area.to_value(u.deg**2),
            },
            {
                "description": "5-sigma NUV limiting magnitude over deepest 7500 deg2",
                "requirement": 25.75,
                "value": np.quantile(
                    coadded_limmags_by_band["NUV"][
                        survey_footprints_by_name["lmlz_wide"]
                    ],
                    np.minimum(
                        1,
                        (
                            7500
                            * u.deg**2
                            / (
                                len(survey_footprints_by_name["lmlz_wide"])
                                * hpx.pixel_area
                            )
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
            {
                "description": "5-sigma FUV limiting magnitude over deepest 7500 deg2",
                "requirement": 25.75,
                "value": np.quantile(
                    coadded_limmags_by_band["FUV"][
                        survey_footprints_by_name["lmlz_wide"]
                    ],
                    np.minimum(
                        1,
                        (
                            7500
                            * u.deg**2
                            / (
                                len(survey_footprints_by_name["lmlz_wide"])
                                * hpx.pixel_area
                            )
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
        ]
    )
)

display(Markdown("## LMLZ Deep"))

display(
    QTable(
        rows=[
            {
                "description": "Area",
                "requirement": 500,
                "value": (visit_map[survey_footprints_by_name["lmlz_deep"]] > 0).sum()
                * hpx.pixel_area.to_value(u.deg**2),
            },
            {
                "description": "5-sigma NUV limiting magnitude over deepest 500 deg2",
                "requirement": 27.0,
                "value": np.quantile(
                    coadded_limmags_by_band["NUV"][
                        survey_footprints_by_name["lmlz_deep"]
                    ],
                    np.minimum(
                        1,
                        (
                            500
                            * u.deg**2
                            / (
                                len(survey_footprints_by_name["lmlz_deep"])
                                * hpx.pixel_area
                            )
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
            {
                "description": "5-sigma FUV limiting magnitude over deepest 500 deg2",
                "requirement": 27.0,
                "value": np.quantile(
                    coadded_limmags_by_band["FUV"][
                        survey_footprints_by_name["lmlz_deep"]
                    ],
                    np.minimum(
                        1,
                        (
                            500
                            * u.deg**2
                            / (
                                len(survey_footprints_by_name["lmlz_deep"])
                                * hpx.pixel_area
                            )
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
        ]
    )
)

display(Markdown("## Magellanic Clouds"))

display(
    QTable(
        rows=[
            {
                "description": "Area",
                "requirement": 50,
                "value": (visit_map[survey_footprints_by_name["mc"]] > 0).sum()
                * hpx.pixel_area.to_value(u.deg**2),
            },
            {
                "description": "20-sigma NUV limiting magnitude over deepest 50 deg2",
                "requirement": 24.5,
                "value": np.quantile(
                    coadded_limmags_by_band["NUV"][survey_footprints_by_name["mc"]],
                    np.minimum(
                        1,
                        (
                            50
                            * u.deg**2
                            / (len(survey_footprints_by_name["mc"]) * hpx.pixel_area)
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
            {
                "description": "20-sigma FUV limiting magnitude over deepest 50 deg2",
                "requirement": 24.5,
                "value": np.quantile(
                    coadded_limmags_by_band["FUV"][survey_footprints_by_name["mc"]],
                    np.minimum(
                        1,
                        (
                            50
                            * u.deg**2
                            / (len(survey_footprints_by_name["mc"]) * hpx.pixel_area)
                        ).to_value(u.dimensionless_unscaled),
                    ),
                ),
            },
        ]
    )
)